# M05 — Stress-Testing Engine

This notebook reproduces M05 from a fresh Google Colab runtime. It runs hypothetical, extreme historical, PCA-factor, and reverse stress tests on the current frozen synthetic Treasury portfolio. Stress losses are conditional scenario results and do not carry forecast probabilities.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JoyWu-302121/market_risk.git'
PROJECT_DIR = Path('/content/market_risk') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB and not (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
elif IN_COLAB:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'src'))
print(f'Project ready at {PROJECT_DIR}')

## Run the accepted M05 pipeline

Reusable code under `src/` constructs and values all scenarios. The runner downloads the official GSW data, preserves failed historical windows, writes reproducible artifacts, and performs the acceptance checks.

In [ ]:
import json

OUTPUT_ROOT = PROJECT_DIR / 'data'
completed = subprocess.run(
    [sys.executable, 'scripts/run_m05_stress_testing.py', '--output-root', str(OUTPUT_ROOT)],
    check=True,
    capture_output=True,
    text=True,
)
report = json.loads(completed.stdout)
assert report['status'] == 'PASS', report['checks']
print(f"M05 status: {report['status']}")
print(f"Valuation date: {report['portfolio']['valuation_date']}")
print(report['scenario_counts'])

## Inspect full-repricing stress results

In [ ]:
import pandas as pd
from IPython.display import display

stress_results = pd.read_csv(report['artifacts']['stress_results_path'])
display(
    stress_results.sort_values('loss', ascending=False)[
        ['scenario_id', 'scenario_family', 'horizon_days', 'loss', 'linear_sensitivity_loss', 'full_minus_linear']
    ].head(15).style.format({
        'loss': '${:,.2f}',
        'linear_sensitivity_loss': '${:,.2f}',
        'full_minus_linear': '${:,.2f}',
    })
)

In [ ]:
import matplotlib.pyplot as plt

hypothetical = stress_results.loc[stress_results['scenario_family'].str.startswith('hypothetical')].copy()
hypothetical = hypothetical.sort_values('loss')
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#b24745' if value > 0 else '#3c78a8' for value in hypothetical['loss']]
ax.barh(hypothetical['scenario_id'], hypothetical['loss'] / 1_000, color=colors)
ax.set(title='Hypothetical Full-Repricing Stress Results', xlabel='Loss (USD thousands)')
ax.axvline(0, color='black', linewidth=0.8)
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

## Inspect PCA loadings and retained residual variance

Factor names remain PC1-PC3 in the engine. Level, slope, and curvature-like interpretations are assigned only after inspecting these loadings.

In [ ]:
pca_loadings = pd.read_csv(report['artifacts']['pca_loadings_path'])
pca_diagnostics = pd.read_csv(report['artifacts']['pca_diagnostics_path'])
display(pca_diagnostics)
print(f"Residual variance after PC1-PC3: {report['pca']['residual_variance_ratio']:.4%}")

fig, ax = plt.subplots(figsize=(10, 5))
for factor in ('PC1', 'PC2', 'PC3'):
    ax.plot(pca_loadings['tenor_years'], pca_loadings[factor], marker='o', label=factor)
ax.set(title='PCA Loadings — Latest 750 Complete Daily Curve Changes', xlabel='Tenor (years)', ylabel='Loading')
ax.axhline(0, color='black', linewidth=0.8)
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Inspect reverse-stress boundaries

Amplitude is the maximum absolute node shock within a normalized shape family. `threshold_not_reached` remains an explicit result.

In [ ]:
reverse_results = pd.read_csv(report['artifacts']['reverse_stress_path'])
display(
    reverse_results[
        ['loss_threshold_usd', 'direction_family', 'status_search', 'amplitude_bps', 'achieved_loss', 'is_minimum_amplitude_for_threshold']
    ].style.format({
        'loss_threshold_usd': '${:,.0f}',
        'amplitude_bps': '{:,.3f}',
        'achieved_loss': '${:,.2f}',
    }, na_rep='—')
)

## Verify M05 completion boundary

In [ ]:
assert all(report['checks'].values())
assert report['scenario_counts']['hypothetical'] == 14
assert report['scenario_counts']['pca'] == 18
assert report['pca']['residual_variance_ratio'] > 0
assert len(report['minimum_reverse_stresses']) == 3
print('M05 stress-testing acceptance checks: PASS')

M05 ends with deterministic scenario losses and conditional reverse-stress thresholds. M06 separately tests rolling VaR exceptions and ES tail behavior, while M07 adds Parametric Normal and stochastic PCA benchmark models.